# 02. 전처리 — 라벨링 + Feature 결합

01_eda.ipynb에서 확인한 사실들을 바탕으로 라벨링(갭필링)과 feature 결합을 직접 구현한다. 팀원 스크립트(`ai/build_and_train_v3_t11.py`, `ai/build_train_table.py` 등)의 로직을 참고하되, 그대로 복붙하지 않고 이해한 내용을 재구현했다.

**v3 수정사항**: Phase 3 ablation 실험(대화에서 A/C/D 세 갈래로 검증) 결과를 반영했다.
- 유동인구(정상판)·KOSIS(세대·인구·사업체) feature는 **모든 변형(절대값/비율변환/행정동명 제거 대체/셀단위)에서 일관되게 역효과**로 확인돼 최종 모델에서는 제외한다(계산 자체는 남겨두어 참고 가능하게 함).
- 대신 **직전 분기 실제 이탈률(모멘텀) feature**가 점포단위·셀단위 모두에서 뚜렷한 개선을 보여 새로 추가했다.
- 셀 단위는 업종 **대분류(10종) 대신 중분류(74종, n≥30 필터)**로 바꾸니 스피어만이 0.32→0.42로 크게 개선됐다 — 최종 셀 단위 그레인을 중분류로 변경.

In [ ]:
import sys
sys.path.insert(0, '.')
import paths
import io, re, zipfile
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)


## 2-1. 라벨링 로직 직접 구현 (갭필링)

"상가업소번호별로 마지막 등장 이후 재등장 없으면 폐업, 짧은 공백(임계값 이내)은 존속으로 채운다"는 규칙을 직접 구현한다. 먼저 원본 21개 분기 스냅샷을 다시 로드한다(01_eda.ipynb의 검증된 로직 재사용).

In [ ]:
USECOLS = [
    "상가업소번호", "시군구코드", "행정동코드", "행정동명",
    "상권업종대분류명", "상권업종중분류명", "상권업종소분류명", "지번주소", "경도", "위도",
]
QUARTER_OVERRIDE = {"20251031": "2025Q3"}


def date8_to_quarter(date8: str) -> str:
    if date8 in QUARTER_OVERRIDE:
        return QUARTER_OVERRIDE[date8]
    y, m = int(date8[:4]), int(date8[4:6])
    return f"{y}Q{(m - 1) // 3 + 1}"


def read_gyeonggi_csv(zf: zipfile.ZipFile, name: str) -> pd.DataFrame:
    return pd.read_csv(
        io.BytesIO(zf.read(name)), encoding="utf-8",
        usecols=lambda c: c in USECOLS,
        dtype={"상가업소번호": str, "시군구코드": str, "행정동코드": str},
    )


def load_snapshot(zip_path) -> pd.DataFrame:
    date8 = re.search(r"(\d{8})", zip_path.stem).group(1)
    quarter = date8_to_quarter(date8)
    with zipfile.ZipFile(zip_path, metadata_encoding="cp949") as z:
        names = z.namelist()
        gg_direct = [n for n in names if "경기" in n and n.endswith(".csv")]
        if gg_direct:
            df = read_gyeonggi_csv(z, gg_direct[0])
        else:
            inner_name = next(n for n in names if n.endswith(".zip"))
            with zipfile.ZipFile(io.BytesIO(z.read(inner_name)), metadata_encoding="cp949") as iz:
                gg_inner = next(n for n in iz.namelist() if "경기" in n and n.endswith(".csv"))
                df = read_gyeonggi_csv(iz, gg_inner)
    df = df[df["시군구코드"] == "41590"].copy()
    df["기준분기"] = quarter
    return df


zip_files = sorted(p for p in paths.SBIZ_DIR.glob("*.zip") if not p.name.startswith("._"))
sbiz = pd.concat([load_snapshot(p) for p in zip_files], ignore_index=True)
quarters = sorted(sbiz["기준분기"].unique(), key=lambda q: (int(q[:4]), int(q[5])))
q_idx = {q: i for i, q in enumerate(quarters)}
sbiz["idx"] = sbiz["기준분기"].map(q_idx)
print("sbiz raw:", sbiz.shape)


In [ ]:
KEEP_COLS = ["상가업소번호", "기준분기", "행정동코드", "행정동명", "상권업종대분류명", "상권업종중분류명",
             "상권업종소분류명", "지번주소", "경도", "위도", "is_filled", "갭길이"]


def build_filled_panel(sbiz: pd.DataFrame, quarters: list, threshold: int) -> pd.DataFrame:
    """점포별로 등장 분기 사이 짧은 공백(<=threshold)을 '존속 중'으로 채운 패널을 만든다."""
    sbiz = sbiz.copy()
    sbiz["is_filled"] = 0
    sbiz["갭길이"] = np.nan
    fill_rows = []
    for store, grp in sbiz.groupby("상가업소번호"):
        grp_sorted = grp.sort_values("idx")
        idxs = grp_sorted["idx"].tolist()
        base_by_idx = {row["idx"]: row for _, row in grp_sorted.iterrows()}
        for a, b in zip(idxs, idxs[1:]):
            gap = b - a - 1
            if 0 < gap <= threshold:
                base = base_by_idx[a]
                for missing_idx in range(a + 1, b):
                    fill_rows.append({
                        "상가업소번호": store, "기준분기": quarters[missing_idx],
                        "행정동코드": base["행정동코드"], "행정동명": base["행정동명"],
                        "상권업종대분류명": base["상권업종대분류명"], "상권업종중분류명": base["상권업종중분류명"],
                        "상권업종소분류명": base["상권업종소분류명"], "지번주소": base["지번주소"],
                        "경도": base["경도"], "위도": base["위도"], "is_filled": 1, "갭길이": gap,
                    })
    fill_df = pd.DataFrame(fill_rows, columns=KEEP_COLS)
    return pd.concat([sbiz[KEEP_COLS], fill_df], ignore_index=True)


panel8 = build_filled_panel(sbiz, quarters, threshold=8)
panel11 = build_filled_panel(sbiz, quarters, threshold=11)
print(f"임계값 8:  채워진 행 {int(panel8['is_filled'].sum()):,} / 전체 {len(panel8):,}")
print(f"임계값 11: 채워진 행 {int(panel11['is_filled'].sum()):,} / 전체 {len(panel11):,}")


**임계값 선택 근거**: 2024Q4/2025Q3에 남아있던 장기갭(9~11분기) 재등장이 임계값 8에서도 안 잡히는지 '개업수(=전분기엔 없다가 이 분기에 생긴 점포 수)' 스파이크로 확인한다.

In [ ]:
def open_counts(sets_by_q: dict, quarters: list) -> dict:
    out = {}
    for i in range(1, len(quarters)):
        q, pq = quarters[i], quarters[i - 1]
        out[q] = len(sets_by_q[q] - sets_by_q[pq])
    return out


orig_sets = {q: set(sbiz.loc[sbiz["기준분기"] == q, "상가업소번호"]) for q in quarters}
p8_sets = {q: set(panel8.loc[panel8["기준분기"] == q, "상가업소번호"]) for q in quarters}
p11_sets = {q: set(panel11.loc[panel11["기준분기"] == q, "상가업소번호"]) for q in quarters}

df_open = pd.DataFrame({
    "원본": open_counts(orig_sets, quarters),
    "임계값8": open_counts(p8_sets, quarters),
    "임계값11": open_counts(p11_sets, quarters),
})
df_open.loc[["2024Q3", "2024Q4", "2025Q1", "2025Q2", "2025Q3"]]


임계값 11에서 2025Q3 잔여 스파이크가 8보다 더 줄어드는 게 확인된다(4,106→2,491→1,443) — **임계값 11을 최종으로 채택**한다(팀원도 동일 결론). 이후 전부 `panel11` 기준으로 진행.

In [ ]:
panel = panel11.copy()
panel["idx"] = panel["기준분기"].map(q_idx)
panel.to_csv(paths.STORE_PANEL_CSV, index=False, encoding="utf-8-sig")
print("저장:", paths.STORE_PANEL_CSV, panel.shape)


## 2-2. 라벨 단위 두 갈래 준비

관측분기 B에서 점포가 (갭필링된 패널 기준) B+1분기에 존재하지 않으면 `label_h1=1`, B+2분기에 존재하지 않으면 `label_h2=1`(팀원 문서와 동일 정의). 주 타깃은 `label_h2`. B+h가 관측 데이터 범위를 벗어나는 마지막 h개 분기는 판정 불가(제외).

In [ ]:
present_by_idx = {i: set(g["상가업소번호"]) for i, g in panel.groupby("idx")}
max_idx = len(quarters) - 1


def build_horizon_label(panel: pd.DataFrame, horizon: int) -> pd.DataFrame:
    """관측분기 B에서 B+horizon 시점에 (갭필링 기준) 존재하지 않으면 1."""
    parts = []
    for idx, grp in panel.groupby("idx"):
        target_idx = idx + horizon
        if target_idx > max_idx:
            continue
        target_set = present_by_idx[target_idx]
        grp = grp.copy()
        grp[f"label_h{horizon}"] = (~grp["상가업소번호"].isin(target_set)).astype(float)
        parts.append(grp)
    return pd.concat(parts, ignore_index=True)


labels_h1 = build_horizon_label(panel, 1)[["상가업소번호", "기준분기", "label_h1"]]
labels_h2 = build_horizon_label(panel, 2)

print(f"label_h1: n={len(labels_h1):,} 양성비율={labels_h1['label_h1'].mean():.2%}")
print(f"label_h2: n={len(labels_h2):,} 양성비율={labels_h2['label_h2'].mean():.2%}")


In [ ]:
store_labels = labels_h2.merge(labels_h1, on=["상가업소번호", "기준분기"], how="left")
print("store_labels:", store_labels.shape)


In [ ]:
store_labels.to_csv(paths.STORE_LABELS_CSV, index=False, encoding="utf-8-sig")
print("저장:", paths.STORE_LABELS_CSV)


## 2-3. Feature 결합

인허가(업력), 유동인구(정상판, share), KOSIS(사업체·인구), R-ONE(정적 그룹), **직전 1분기 실제 이탈률(모멘텀)**을 결합한다.

**카드매출은 제외**(매핑표 부재 + 팀원 ablation에서 효용 없음 확인, 이전과 동일).

### 2-3-1. 인허가 매칭 (업력)

지번주소를 정규화(`norm_addr`)해 완전일치로 연결한다.

In [ ]:
def norm_addr(s) -> str:
    if pd.isna(s):
        return ""
    s = re.sub(r"경기도|화성시|효행구|만세구|동탄구|병점구", "", str(s))
    m = re.search(r"\d+(-\d+)?", s)
    if m:
        s = s[:m.end()]
    return "".join(s.split())


permit_files = sorted(p for p in paths.PERMIT_DIR.glob("*.csv") if not p.name.startswith("._"))
permit_frames = []
for p in permit_files:
    df = None
    for enc in ["cp949", "utf-8"]:
        try:
            df = pd.read_csv(p, encoding=enc, low_memory=False)
            break
        except UnicodeDecodeError:
            continue
    if df is None or "지번주소" not in df.columns or "인허가일자" not in df.columns:
        continue
    permit_frames.append(df[["지번주소", "인허가일자"]])

permit_all = pd.concat(permit_frames, ignore_index=True)
permit_all["지번주소_norm"] = permit_all["지번주소"].map(norm_addr)
permit_all["인허가일자_dt"] = pd.to_datetime(permit_all["인허가일자"], format="%Y-%m-%d", errors="coerce")
permit_all = permit_all.dropna(subset=["인허가일자_dt"])
permit_lookup = (permit_all.sort_values("인허가일자_dt")
                 .drop_duplicates(subset=["지번주소_norm"], keep="first")
                 .set_index("지번주소_norm")["인허가일자_dt"])

store_addr = store_labels[["상가업소번호", "지번주소"]].drop_duplicates("상가업소번호").copy()
store_addr["지번주소_norm"] = store_addr["지번주소"].map(norm_addr)
store_addr["permit_qoffset"] = store_addr["지번주소_norm"].map(permit_lookup).apply(
    lambda d: (d.year * 4 + (d.month - 1) // 3 + 1) if pd.notna(d) else np.nan
)
print(f"인허가 매칭률(점포 기준): {store_addr['permit_qoffset'].notna().mean():.1%}")
age_lookup = store_addr.set_index("상가업소번호")[["permit_qoffset"]]


### 2-3-2. 유동인구(정상판) share / KOSIS / R-ONE 정적 그룹

⚠️ **아래 세 feature는 계산은 하되, ablation 결과(대화 참고) 전부 역효과로 확인돼 최종 모델에서는 제외한다.** 계산 자체는 참고용으로 `store_train_table.csv`에 남겨둔다.

In [ ]:
dong_list = pd.read_csv(paths.GYEONGGI_DONG_LIST_CSV, encoding="cp949", dtype={"읍면동코드": str})
hw_dong = dong_list[
    dong_list["상세주소"].str.contains("화성시", na=False)
    & ~dong_list["읍면동명"].isin(["화성시동탄출장소", "화성시동부출장소", "화성시"])
]
code_to_name = hw_dong.set_index("읍면동코드")["읍면동명"]

fpop = pd.read_csv(paths.FLOATING_POP_CSV, encoding="utf-8", dtype={"ADMDONG_CD": str})
fpop = fpop[fpop["WDAY_CD"] == "TOT"].copy()
fpop["행정동명"] = fpop["ADMDONG_CD"].map(code_to_name)
fpop["연"] = fpop["STD_YM"] // 100
fpop["월"] = fpop["STD_YM"] % 100
fpop["기준분기"] = fpop["연"].astype(str) + "Q" + (((fpop["월"] - 1) // 3) + 1).astype(str)
fpop_q = fpop.groupby(["행정동명", "기준분기"], as_index=False)["DYNMC_POPLTN_CNT"].mean()
fpop_q["유동인구_share"] = fpop_q["DYNMC_POPLTN_CNT"] / fpop_q.groupby("기준분기")["DYNMC_POPLTN_CNT"].transform("sum")


def parse_kosis_pop(path) -> pd.DataFrame:
    raw = pd.read_csv(path, encoding="cp949", header=None)
    years_row, item_row, unit_row, gender_row = raw.iloc[0], raw.iloc[1], raw.iloc[2], raw.iloc[3]
    data = raw.iloc[4:].reset_index(drop=True)
    records = []
    for col in range(1, raw.shape[1]):
        if item_row[col] == "세대 수 (세대)":
            key = "세대수"
        elif item_row[col] == "등록인구 (명)" and unit_row[col] == "합계" and gender_row[col] == "소계":
            key = "등록인구"
        else:
            continue
        for row_i in range(len(data)):
            dong = data.iloc[row_i, 0]
            if dong == "합계":
                continue
            records.append({"행정동명": dong, "연도": years_row[col], key: pd.to_numeric(data.iloc[row_i, col], errors="coerce")})
    return pd.DataFrame(records).pivot_table(index=["행정동명", "연도"], values=["세대수", "등록인구"], aggfunc="first").reset_index()


def parse_kosis_biz(path) -> pd.DataFrame:
    raw = pd.read_csv(path, encoding="cp949", header=None)
    years_row, industry_row, item_row, sub_row = raw.iloc[0], raw.iloc[1], raw.iloc[2], raw.iloc[3]
    data = raw.iloc[4:].reset_index(drop=True)
    records = []
    for col in range(1, raw.shape[1]):
        if industry_row[col] == "합계" and item_row[col] == "사업체수 (개)" and sub_row[col] == "소계":
            key = "사업체수"
        elif industry_row[col] == "합계" and item_row[col] == "종사자수 (명)" and sub_row[col] == "계":
            key = "종사자수"
        else:
            continue
        for row_i in range(len(data)):
            dong = data.iloc[row_i, 0]
            if dong == "합계":
                continue
            records.append({"행정동명": dong, "연도": years_row[col], key: pd.to_numeric(data.iloc[row_i, col], errors="coerce")})
    return pd.DataFrame(records).pivot_table(index=["행정동명", "연도"], values=["사업체수", "종사자수"], aggfunc="first").reset_index()


kosis_pop = parse_kosis_pop(paths.KOSIS_HOUSEHOLD_POP_CSV)
kosis_biz = parse_kosis_biz(paths.KOSIS_BUSINESS_CSV)
kosis = kosis_pop.merge(kosis_biz, on=["행정동명", "연도"], how="outer")
kosis["연도"] = kosis["연도"].astype(int)

DONGTAN_GROUP = [f"동탄{n}동" for n in range(1, 10)]
BYEONGJEOM_GROUP = ["병점1동", "병점2동", "화산동", "진안동"]


def rent_group(dong: str) -> str:
    if dong in DONGTAN_GROUP:
        return "동탄권"
    if dong in BYEONGJEOM_GROUP:
        return "병점권"
    return "기타"


print("유동인구/KOSIS/R-ONE 계산 완료(참고용, 최종 모델 feature에서는 제외)")


### 2-3-3. 직전 1분기 실제 이탈률 (모멘텀 feature) — **채택**

행정동×상권업종대분류명 셀에서, 직전 분기(B-1)에 있던 점포 중 이번 분기(B)에 사라진 비율. 미래 정보를 전혀 안 쓰고 관측분기 B 시점에 이미 확정된 값만 쓰므로 시간 누수가 없다. 대화에서 확인한 ablation 결과: 점포단위 PR-AUC 0.1344→0.1356, 셀단위(대분류) 스피어만 0.3232→0.3416 — 유일하게 일관되게 도움이 된 신규 feature.

In [ ]:
prev_sets_daebun = {(dong, cat, idx): set(g["상가업소번호"])
                    for (dong, cat, idx), g in panel.groupby(["행정동명", "상권업종대분류명", "idx"])}


def departure_rate(dong, cat, idx):
    if idx == 0:
        return np.nan
    prev = prev_sets_daebun.get((dong, cat, idx - 1), set())
    if len(prev) == 0:
        return np.nan
    cur = prev_sets_daebun.get((dong, cat, idx), set())
    return len(prev - cur) / len(prev)


cell_keys = panel[["행정동명", "상권업종대분류명"]].drop_duplicates()
grid = cell_keys.merge(pd.DataFrame({"idx": range(len(quarters))}), how="cross")
grid["최근1분기이탈률"] = grid.apply(lambda r: departure_rate(r["행정동명"], r["상권업종대분류명"], r["idx"]), axis=1)
idx_to_q = {i: q for q, i in q_idx.items()}
grid["기준분기"] = grid["idx"].map(idx_to_q)
momentum_lookup = grid[["행정동명", "상권업종대분류명", "기준분기", "최근1분기이탈률"]]
print("모멘텀 feature 테이블:", momentum_lookup.shape)


### 2-3-4. 최종 병합 — `store_train_table`

In [ ]:
def qoffset_from_quarter(q: str) -> int:
    y, qn = int(q[:4]), int(q[5])
    return y * 4 + qn


store_train = store_labels.merge(age_lookup, on="상가업소번호", how="left")
store_train["관측분기_qoffset"] = store_train["기준분기"].map(qoffset_from_quarter)
store_train["업력_분기수"] = store_train["관측분기_qoffset"] - store_train["permit_qoffset"]

store_train = store_train.merge(fpop_q[["행정동명", "기준분기", "유동인구_share"]], on=["행정동명", "기준분기"], how="left")
store_train["연도"] = store_train["기준분기"].str[:4].astype(int).clip(upper=2023)
store_train = store_train.merge(kosis, on=["행정동명", "연도"], how="left")
store_train["임대료_매핑그룹"] = store_train["행정동명"].map(rent_group)
store_train = store_train.merge(momentum_lookup, on=["행정동명", "상권업종대분류명", "기준분기"], how="left")

print("store_train_table:", store_train.shape)
for c in ["업력_분기수", "유동인구_share", "세대수", "등록인구", "사업체수", "종사자수", "최근1분기이탈률"]:
    print(f"  {c} 결측률: {store_train[c].isna().mean():.1%}")


In [ ]:
store_train.to_csv(paths.STORE_TRAIN_TABLE_CSV, index=False, encoding="utf-8-sig")
print("저장:", paths.STORE_TRAIN_TABLE_CSV, store_train.shape)


### 2-3-5. `cell_train_table` — **행정동×업종중분류(74종)×분기** 집계

대화의 ablation에서 대분류(10종)보다 중분류(74종, n≥30 필터)가 스피어만 0.32→0.42로 뚜렷이 나아서 그레인을 중분류로 바꾼다. 필터(n≥30)는 저장 시점이 아니라 Phase 3 학습 시점에 적용(원본 그대로 보존).

In [ ]:
cell_train = (store_train.groupby(["행정동명", "상권업종중분류명", "기준분기"])
              .agg(점포수=("상가업소번호", "nunique"),
                   폐업률=("label_h2", "mean"),
                   평균업력_분기수=("업력_분기수", "mean"),
                   임대료_매핑그룹=("임대료_매핑그룹", "first"))
              .reset_index())
print("cell_train_table(중분류):", cell_train.shape)
cell_train.to_csv(paths.CELL_TRAIN_TABLE_CSV, index=False, encoding="utf-8-sig")
print("저장:", paths.CELL_TRAIN_TABLE_CSV)


## 2-4. 시간 누수 체크리스트

- [x] **무작위 분할 금지** — Phase 3에서 시간순 분할
- [x] **시간 대리변수 배제** — R-ONE 원본 수치 대신 정적 그룹만(단, 최종 모델엔 이마저 제외)
- [x] **업력 좌측절단 보정** — 인허가일자 기준
- [x] **모멘텀 feature 시간 누수 점검** — `최근1분기이탈률`은 B-1→B 구간만 사용(관측분기 B 시점에 이미 확정된 값), B+1/B+2를 넘겨보지 않음 — 라벨(`label_h2`, B+2 시점)과 시점이 겹치지 않음 확인
- [ ] **결측 시기 편중 확인** — `유동인구_share`가 2025Q3/Q4 결측(참고용 feature라 최종 모델엔 미포함이라 영향 없음)

**유동인구/KOSIS를 최종 제외한 이유**: 대화에서 A(비율/밀도 변환)·C(행정동명 제거 후 대체)·D(셀단위) 세 방향으로 다 시도했으나 전부 "제외" 버전을 못 이김 — 심지어 행정동명과 같이 넣었을 때가 행정동명·신규데이터 둘 다 뺀 경우보다도 나빠서(노이즈로 작용) 최종 제외 확정.

## 산출물 요약

- `data/processed/store_panel.csv` — 갭필링 적용된 점포×분기 패널(임계값 11)
- `data/processed/store_labels.csv` — 라벨(`label_h1`, `label_h2`)
- `data/processed/store_train_table.csv` — feature 결합 완료(최근1분기이탈률 포함, 유동인구/KOSIS는 참고용으로만 포함)
- `data/processed/cell_train_table.csv` — 행정동×업종**중분류**×분기 집계

Phase 3에서 점포단위(업력+최근1분기이탈률+카테고리)와 셀단위(중분류, n≥30) 최종 모델을 학습한다.